# Comparing three 2019 Alaska fires

Shovel Creek, Swan Lake and Nugget Creek were selected for existing work, geographic contrast and field-reference prospects, before examining their transfer scores. This notebook uses the existing pipeline and preserves the separate Shovel Creek experiment in notebook 01.

- **Question:** Does the same retrospective spectral-recovery model work within and between these fires?
- **Inputs:** Alaska Fire History, NASA HLS L30/S30, Dynamic World, USGS 3DEP with Copernicus fallback, TerraClimate; MTBS event metadata supplies reference context. Exact asset IDs and event records are saved locally.
- **Limits:** Three deliberately selected fires from one year cannot establish Alaska-wide accuracy. NDVI change is not forest recovery, and retrospective climate averages do not support forecasting claims. Notebook 03 provides an untuned selected-plot severity comparison; ecological recovery validation remains outstanding.
- **Design:** 2018 summer baseline, 2020 severity assessment, 2020–2024 and 2021–2024 NDVI slopes. HLS/Dynamic World use June 15 inclusive–August 15 exclusive; climate uses JJA relative to 1991–2020.
- **Execution:** Run from the notebook directory or repository root using `fireaidenv`. Samples and provenance are saved outside Git in `../validation/multi_fire_comparison`. Save notebook edits before running: the cache fingerprint uses the saved acquisition cells. A matching cache is reused with lossless float parsing; use a fresh output directory to change inputs or refresh remote imagery.

In [1]:
from pathlib import Path
import hashlib
import json
import re
import sys
from datetime import datetime, timezone

repo = Path.cwd()
if not (repo / 'src').is_dir():
    repo = repo.parent if (repo.parent / 'src').is_dir() else repo / 'fireaid-burn-severity-recovery'
assert (repo / 'src/config.py').is_file(), 'Run from the notebook directory or repository root'
sys.path.insert(0, str(repo))
import ee
import geemap
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from src.auth import init_ee
from src.config import ASSETS, GEE_PROJECT, SEVERITY_BINS, SEVERITY_CLASSES, DNBR_RAMP
from src.imagery import get_peak_summer_hls
from src.severity import compute_burn_severity
from src.terrain import get_terrain_covariates
from src.landcover import get_prefire_landcover
from src.timeseries import get_multiyear_recovery_stack, get_terraclimate_anomalies
from src.modeling import sample_fire_scar, calculate_recovery_slopes, fit_recovery_ols, buffered_spatial_cv, compute_morans_i

init_ee(GEE_PROJECT)
ee.data.setDeadline(180000)
output_dir = repo.parent / 'validation' / 'multi_fire_comparison'
output_dir.mkdir(parents=True, exist_ok=True)
years = list(range(2020, 2025))
predictors = ['dNBR', 'elevation', 'slope', 'northness', 'trees', 'temp_anomaly', 'precip_anomaly']
predictor_sets = {'full': predictors, 'initial_NDVI_only': ['NDVI_2020'],
                  'full_plus_initial_NDVI': [*predictors, 'NDVI_2020']}
cases = {
    'Shovel Creek': dict(event_id='AK6498714841520190621', crs='EPSG:32606'),
    'Swan Lake': dict(event_id='AK6063115043820190605', crs='EPSG:32605'),
    'Nugget Creek': dict(event_id='AK6487414655520190621', crs='EPSG:32606'),
}
print('Provisional sampling bins (not MTBS classes):', SEVERITY_BINS)
print('Local artifacts:', output_dir)

[auth] Earth Engine initialized successfully with project: epscor-ioc
Provisional sampling bins (not MTBS classes): [-100, 100, 270, 440, 660]
Local artifacts: /mnt/d/uaa/fireaid/burn severity repo/validation/multi_fire_comparison


## 1. Match events, check inputs, and sample each fire

The same provisional dNBR bins define the sampling strata for every fire. MTBS thresholds/dates are recorded separately; they do not replace these bins, and metadata matching is not reference accuracy assessment.

Sampling uses a 200 m nominal grid in each fire's UTM CRS, seed 0, and a target of 2,000 points split across six classes. Rare classes may supply fewer points. Values are grid evaluations, not 200 m area averages or guaranteed independent observations. All metrics weight sampled points equally, not land area. Aggregation `tileScale=4` controls memory without changing the requested sampling resolution.

Aggregation control: [Earth Engine stratifiedSample documentation](https://developers.google.com/earth-engine/apidocs/ee-image-stratifiedsample).


In [2]:
source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (repo / 'src').glob('*.py')}
notebook_path = repo / 'notebooks/02_fire_comparison.ipynb'
notebook_json = json.loads(notebook_path.read_text())
acquisition_hash = hashlib.sha256(''.join(''.join(c['source']) for c in notebook_json['cells']
    if 'acquisition' in c.get('metadata', {}).get('tags', [])).encode()).hexdigest()

samples, records, layers = {}, {}, {}
for name, case in cases.items():
    matches = ee.FeatureCollection(ASSETS['ak_fire_history']).filter(ee.Filter.eq('FIREYEAR', '2019')).filter(ee.Filter.eq('NAME', name))
    assert matches.size().getInfo() == 1, f'Ambiguous perimeter: {name}'
    fire = ee.Feature(matches.first())
    props = fire.toDictionary().getInfo()
    geom = fire.geometry()
    events = ee.FeatureCollection(ASSETS['mtbs']).filter(ee.Filter.eq('Event_ID', case['event_id']))
    assert events.size().getInfo() == 1, f'Ambiguous MTBS event: {name}'
    reference = ee.Feature(events.first()).toDictionary().getInfo()
    assert props['IRWINID'].strip('{}').lower() == reference['irwinID'].strip('{}').lower(), name
    projection = ee.Projection(case['crs']).atScale(30)
    pre = get_peak_summer_hls(geom, 2018).setDefaultProjection(projection)
    post = get_peak_summer_hls(geom, 2020).setDefaultProjection(projection)
    severity = compute_burn_severity(pre, post)
    stack = ee.Image([severity, get_terrain_covariates(geom), get_prefire_landcover(geom, 2018),
        get_terraclimate_anomalies(geom, 2020, 2024), get_multiyear_recovery_stack(geom, 2020, 2024)])
    layers[name] = dict(fire=fire, severity=severity)
    recipe = dict(fire=props, mtbs=reference, case=case, assets=ASSETS, source_sha256=source_hashes,
        acquisition_sha256=acquisition_hash, sample_scale_m=200, analysis_scale_m=30,
        target_points=2000, seed=0, tile_scale=4, pre_year=2018, recovery_years=years)
    slug = name.lower().replace(' ', '_')
    csv_path, record_path = output_dir / f'{slug}_samples.csv', output_dir / f'{slug}.json'
    if record_path.exists():
        record = json.loads(record_path.read_text())
        assert record['recipe'] == recipe, f'{name}: cached inputs changed; use a fresh output_dir'
        assert record['sample_sha256'] == hashlib.sha256(csv_path.read_bytes()).hexdigest(), f'{name}: sample checksum mismatch'
        frame = pd.read_csv(csv_path, float_precision='round_trip')
        print(name, 'using verified sample cache', flush=True)
    else:
        valid = stack.mask().reduce(ee.Reducer.min()).unmask(0, sameFootprint=False)
        area = ee.Image.pixelArea()
        support = area.rename('total_m2').addBands(area.multiply(valid).rename('valid_m2')).reduceRegion(
            ee.Reducer.sum(), geometry=geom, crs=projection, scale=30, tileScale=4, maxPixels=100_000_000).getInfo()
        assert 0 < support['valid_m2'] <= support['total_m2'], f'{name}: no usable stack'
        print(name, '30 m stack coverage', support['valid_m2'] / support['total_m2'], flush=True)
        frame = sample_fire_scar(stack, geom, scale=200, max_points=2000, projection=projection, tile_scale=4)
        required = [*predictors, *[f'NDVI_{y}' for y in years], 'easting', 'northing', 'longitude', 'latitude']
        assert len(frame) >= 10 and np.isfinite(frame[required].to_numpy(dtype=float)).all(), name
        frame.to_csv(csv_path, index=False)
        record = dict(recipe=recipe, acquired_utc=datetime.now(timezone.utc).isoformat(), support_30m=support,
            sample_sha256=hashlib.sha256(csv_path.read_bytes()).hexdigest())
        record_path.write_text(json.dumps(record, indent=2) + '\n')
    assert not frame[['easting', 'northing']].duplicated().any(), f'{name}: duplicate grid points'
    for coordinate in ['easting', 'northing']:
        assert np.allclose(frame[coordinate] % 200, 100, atol=0.01, rtol=0), f'{name}: incorrect grid units'
    for year in years:
        assert frame[f'NDVI_{year}'].between(-1, 1).all(), f'{name}: NDVI outside bounds'
    assert frame.severity_class.isin(range(6)).all(), name
    samples[name], records[name] = frame, record
    print(name, 'samples:', len(frame), 'class counts:', frame.severity_class.value_counts().sort_index().to_dict(), flush=True)

context_rows = []
for name, record in records.items():
    ref = record['recipe']['mtbs']
    dates = [re.search(r'(\d{8})(?:_30m)?$', ref[key]).group(1) for key in ['Pre_ID', 'Post_ID']]
    support = record['support_30m']
    xy = samples[name][['easting', 'northing']].to_numpy()
    lonlat = samples[name][['longitude', 'latitude']].to_numpy()
    native_cells = np.floor(np.column_stack(((lonlat[:, 0] + 180) * 24, (90 - lonlat[:, 1]) * 24)))
    context_rows.append(dict(fire=name, samples=len(samples[name]), coverage_30m=support['valid_m2']/support['total_m2'],
        blocks_2km=len(np.unique(np.floor(xy/2000), axis=0)), blocks_4km=len(np.unique(np.floor(xy/4000), axis=0)),
        climate_cells_with_samples=len(np.unique(native_cells, axis=0)), crs=cases[name]['crs'],
        mtbs_pre=dates[0], mtbs_post=dates[1], mtbs_high=ref['High_T'], mtbs_offset=ref['dNBR_offst']))
case_context = pd.DataFrame(context_rows)
print(case_context.to_string(index=False))
case_context.to_csv(output_dir / 'case_context.csv', index=False)

Shovel Creek 30 m stack coverage 0.9936700229197142
Shovel Creek samples: 1573 class counts: {1: 278, 2: 296, 3: 333, 4: 333, 5: 333}
Swan Lake 30 m stack coverage 0.9848845001749221
Swan Lake samples: 1708 class counts: {0: 43, 1: 333, 2: 333, 3: 333, 4: 333, 5: 333}
Nugget Creek 30 m stack coverage 0.9953891734604419
Nugget Creek samples: 1053 class counts: {1: 120, 2: 218, 3: 333, 4: 333, 5: 49}
        fire  samples  coverage_30m  blocks_2km  blocks_4km  climate_cells_with_samples        crs mtbs_pre mtbs_post  mtbs_high  mtbs_offset
Shovel Creek     1573      0.993670          48          17                          28 EPSG:32606 20180722  20200609        700           14
   Swan Lake     1708      0.984885         208          64                          91 EPSG:32605 20170806  20200728        660           36
Nugget Creek     1053      0.995389          31          11                          19 EPSG:32606 20180929  20200817        615           20


## 2. Within-fire spatial validation

Retain all 2/4 km block × 0/1/2 km buffer designs and both response windows. Compare the existing full model, initial NDVI alone, and the full model plus initial NDVI; do not select a winner from these exploratory scores. The initial-NDVI models share a year with the 2020–2024 response. Failed designs are reported explicitly. Block/buffer distances are sensitivity choices, not estimated independence ranges.

In [3]:
def score_predictions(observed, predicted, baseline):
    observed, predicted, baseline = [np.asarray(v, dtype=float) for v in [observed, predicted, baseline]]
    assert observed.shape == predicted.shape == baseline.shape and np.isfinite([observed, predicted, baseline]).all()
    errors = observed - predicted
    tss = np.sum((observed - observed.mean())**2)
    assert tss > 0, 'Constant response cannot supply R-squared'
    return dict(rmse=float(np.sqrt(np.mean(errors**2))),
        baseline_rmse=float(np.sqrt(np.mean((observed-baseline)**2))),
        pooled_r2=float(1-np.sum(errors**2)/tss))

within_rows, within_predictions, within_folds, diagnostics = [], [], [], []
for name, frame in samples.items():
    for start in [2020, 2021]:
        data = frame.copy()
        data['recovery_slope'] = calculate_recovery_slopes(data, [y for y in years if y >= start])
        fit, vif = fit_recovery_ols(data, predictors)
        moran = compute_morans_i(fit.resid.to_numpy(), data[['longitude', 'latitude']].to_numpy())
        diagnostics.append(dict(fire=name, start_year=start, in_sample_r2=fit.rsquared, **moran))
        for block in [2000, 4000]:
            for buffer in [0, 1000, 2000]:
                expected_folds = None
                for label, variables in predictor_sets.items():
                    identity = dict(fire=name, start_year=start, block_m=block, buffer_m=buffer, model=label)
                    try:
                        pred, folds = buffered_spatial_cv(data, variables, block, buffer)
                    except ValueError as error:
                        within_rows.append(dict(**identity, status='failed', reason=str(error)))
                        continue
                    if expected_folds is not None:
                        pd.testing.assert_frame_equal(folds, expected_folds)
                    expected_folds = folds
                    assert folds.min_train_test_m.min() >= buffer
                    within_rows.append(dict(**identity, status='ok', n=len(pred), blocks=len(folds),
                        min_training_rows=int(folds.n_train.min()), **score_predictions(pred.observed, pred.predicted, pred.baseline)))
                    within_predictions.append(pred.rename_axis('sample_id').reset_index().assign(**identity))
                    within_folds.append(folds.assign(**identity))
        print(name, start, 'spatial designs complete', flush=True)
within_scores = pd.DataFrame(within_rows)
within_predictions = pd.concat(within_predictions, ignore_index=True)
within_folds = pd.concat(within_folds, ignore_index=True)
diagnostics = pd.DataFrame(diagnostics)
for filename, table in [('within_scores', within_scores), ('within_predictions', within_predictions),
                        ('within_folds', within_folds), ('residual_diagnostics', diagnostics)]:
    table.to_csv(output_dir / f'{filename}.csv', index=False)
print(within_scores.groupby(['fire', 'start_year', 'model']).pooled_r2.agg(['min', 'max']).to_string())
print('Failed designs:', within_scores.loc[within_scores.status != 'ok'].to_string(index=False))
print('Residual dependence; ordinary OLS p-values are not validated:')
print(diagnostics.to_string(index=False))

Shovel Creek 2020 spatial designs complete
Shovel Creek 2021 spatial designs complete
Swan Lake 2020 spatial designs complete
Swan Lake 2021 spatial designs complete
Nugget Creek 2020 spatial designs complete
Nugget Creek 2021 spatial designs complete
                                                     min       max
fire         start_year model                                     
Nugget Creek 2020       full                    0.129592  0.299305
                        full_plus_initial_NDVI  0.208953  0.438437
                        initial_NDVI_only       0.313773  0.396606
             2021       full                    0.297422  0.343082
                        full_plus_initial_NDVI  0.317849  0.385646
                        initial_NDVI_only       0.367820  0.397705
Shovel Creek 2020       full                    0.500504  0.527190
                        full_plus_initial_NDVI  0.590644  0.633255
                        initial_NDVI_only       0.576697  0.595126
           

## 3. Leave one fire out

Fit on the other two fires and predict the entire held-out fire. Neither its response nor its feature distribution controls training or feature selection. The baseline is the pooled training sample's response mean. Local UTM coordinates are **not** pooled into a distance calculation. Report all three predictor sets and both response windows.

Training-range checks flag marginal predictor extrapolation; being inside every marginal range does not guarantee multivariate support. These are retrospective transfer checks on three selected fires, not independent evidence for a statewide model. A high within-fire score can coexist with poor transfer.

In [4]:
transfer_rows, transfer_predictions, range_rows = [], [], []
for start in [2020, 2021]:
    data = {name: frame.assign(recovery_slope=calculate_recovery_slopes(frame, [y for y in years if y >= start]))
            for name, frame in samples.items()}
    for held_out in cases:
        training_names = [name for name in cases if name != held_out]
        training = pd.concat([data[name] for name in training_names], ignore_index=True)
        test = data[held_out]
        for label, variables in predictor_sets.items():
            fit, _ = fit_recovery_ols(training, variables)
            active = [v for v in fit.model.exog_names if v != 'const']
            predicted = fit.predict(sm.add_constant(test[active], has_constant='add')).to_numpy()
            baseline = np.full(len(test), training.recovery_slope.mean())
            identity = dict(fire=held_out, start_year=start, model=label)
            outside = np.zeros(len(test), dtype=bool)
            for variable in active:
                lower, upper = training[variable].min(), training[variable].max()
                out = ~test[variable].between(lower, upper).to_numpy()
                outside |= out
                range_rows.append(dict(**identity, variable=variable, training_min=lower, training_max=upper,
                    test_min=test[variable].min(), test_max=test[variable].max(), fraction_outside=float(out.mean())))
            transfer_rows.append(dict(**identity, n_train=len(training), n_test=len(test),
                training_fires='; '.join(training_names), active_predictors='; '.join(active),
                fraction_outside_any_training_range=float(outside.mean()),
                **score_predictions(test.recovery_slope, predicted, baseline)))
            transfer_predictions.append(pd.DataFrame(dict(sample_id=test.index, observed=test.recovery_slope,
                predicted=predicted, baseline=baseline)).assign(**identity))
transfer_scores = pd.DataFrame(transfer_rows)
transfer_predictions = pd.concat(transfer_predictions, ignore_index=True)
transfer_ranges = pd.DataFrame(range_rows)
for filename, table in [('transfer_scores', transfer_scores), ('transfer_predictions', transfer_predictions), ('transfer_ranges', transfer_ranges)]:
    table.to_csv(output_dir / f'{filename}.csv', index=False)
print(transfer_scores.to_string(index=False))

        fire  start_year                  model  n_train  n_test             training_fires                                                                 active_predictors  fraction_outside_any_training_range     rmse  baseline_rmse  pooled_r2
Shovel Creek        2020                   full     2761    1573    Swan Lake; Nugget Creek            dNBR; elevation; slope; northness; trees; temp_anomaly; precip_anomaly                             0.001907 0.016860       0.016711  -0.019669
Shovel Creek        2020      initial_NDVI_only     2761    1573    Swan Lake; Nugget Creek                                                                         NDVI_2020                             0.000000 0.011115       0.016711   0.556825
Shovel Creek        2020 full_plus_initial_NDVI     2761    1573    Swan Lake; Nugget Creek dNBR; elevation; slope; northness; trees; temp_anomaly; precip_anomaly; NDVI_2020                             0.001907 0.017234       0.016711  -0.065414
   Swan Lake    

## 4. Explore the results

The spatial-score spread reflects alternative validation designs, not confidence intervals. Trajectories show class-wise sample means, not fire-wide area means. The map shows provisional dNBR; its colors are not field-calibrated severity classes. Browser/widget rendering requires an interactive notebook session.

In [5]:
fig_within = px.scatter(within_scores.query("status == 'ok'"), x='fire', y='pooled_r2', color='model',
    symbol='block_m', facet_col='start_year', hover_data=['buffer_m', 'rmse', 'baseline_rmse'],
    title='Within-fire prediction: every spatial design retained')
fig_within.add_hline(y=0, line_dash='dash')
fig_within.show()
fig_transfer = px.scatter(transfer_scores, x='fire', y='pooled_r2', color='model', facet_col='start_year',
    hover_data=['rmse', 'baseline_rmse', 'fraction_outside_any_training_range'],
    title='Held-out-fire prediction: trained on the other two fires')
fig_transfer.add_hline(y=0, line_dash='dash')
fig_transfer.show()
trajectory = pd.concat([frame.groupby('severity_class')[[f'NDVI_{y}' for y in years]].mean().reset_index().assign(fire=name)
                        for name, frame in samples.items()], ignore_index=True)
trajectory = trajectory.melt(id_vars=['fire', 'severity_class'], var_name='year', value_name='sample_mean_ndvi')
trajectory['year'] = trajectory.year.str.replace('NDVI_', '').astype(int)
trajectory['class'] = trajectory.severity_class.map(dict(enumerate(SEVERITY_CLASSES)))
fig_trajectory = px.line(trajectory, x='year', y='sample_mean_ndvi', color='class', facet_col='fire', markers=True,
    title='Class-wise sampled NDVI: provisional bins, not area-weighted fire means')
fig_trajectory.show()
m = geemap.Map(center=[63, -149], zoom=5)
for name, layer in layers.items():
    m.addLayer(layer['severity'].select('dNBR'), {'min': -100, 'max': 800, 'palette': DNBR_RAMP}, f'{name}: dNBR')
    m.addLayer(ee.Image().paint(ee.FeatureCollection([layer['fire']]), 1, 2), {'palette': ['FFFFFF']}, f'{name}: perimeter')
m

Figure saved: figure_1.html; browser rendering unverified
Figure saved: figure_2.html; browser rendering unverified
Figure saved: figure_3.html; browser rendering unverified


## Interpretation checkpoint

- Compare held-out-fire errors with the training-mean baseline and predictor-range flags before making transfer claims.
- Inspect residual dependence and response-window sensitivity; ordinary OLS significance remains exploratory.
- Preserve failed designs and poor scores. Do not retune using held-out outcomes and then label them independent validation.
- Field-reference checks in notebook 03 and date-aligned MTBS comparisons are separate from recovery-model validation.

Saved September 21 run, independently checked September 26: all 108 spatial designs and 18 transfer comparisons completed. Full-model transfer R² for the 2020/2021 response starts is −0.020/−1.144 (Shovel), 0.314/0.102 (Swan), and 0.027/0.233 (Nugget). These are preliminary results from the saved run, not guarantees for changed inputs. The saved app and HTML viewer have been checked in a browser. Selected-plot field checks are described in notebook 03; fire-wide severity accuracy and ecological recovery remain unestablished.
